In [0]:
# Databricks notebook source

# ============================================
# PARAMETRO DE ENV VINDO DA PIPELINE / JOB
# ============================================
dbutils.widgets.text("env", "")
p_env = dbutils.widgets.get("env")

env_norm = p_env.strip().lower()
if env_norm in ("hmg", "homol", "hml"):
    env_suffix = "hml"
elif env_norm in ("prod", "prd"):
    env_suffix = "prd"
else:
    env_suffix = "dev"

control_catalog = f"platform_{env_suffix}"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {control_catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {control_catalog}.utils")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {control_catalog}.control")

spark.sql(f"""
CREATE OR REPLACE PROCEDURE {control_catalog}.utils.proc_load_config
(
  IN p_config_name STRING,
  IN p_layer       STRING,
  IN p_system      STRING
)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN
  DECLARE v_config_name STRING;
  DECLARE v_layer STRING;
  DECLARE v_system STRING;
  DECLARE v_cfg_count INT;
  DECLARE v_sql STRING;

  SET v_config_name = LOWER(TRIM(p_config_name));
  SET v_layer       = UPPER(TRIM(p_layer));
  SET v_system      = LOWER(TRIM(p_system));

  SET v_sql =
    "CREATE OR REPLACE TEMP VIEW cfg AS
     WITH constraints_agg AS (
       SELECT
         const.config_id,
         ARRAY_AGG(STRUCT(const.*)) AS cfg_constraints
       FROM {control_catalog}.control.cfg_table_constraints const
       WHERE const.is_active = TRUE
       GROUP BY const.config_id
     ),
     state_ranked AS (
       SELECT
         state.*,
         ROW_NUMBER() OVER (
           PARTITION BY state.config_id
           ORDER BY COALESCE(state.last_run_ts, TIMESTAMP('1900-01-01 00:00:00')) DESC
         ) AS rn
       FROM {control_catalog}.control.cfg_ingestion_state state
       WHERE state.state_type = 'WATERMARK'
     ),
     state_agg AS (
       SELECT
         config_id,
         STRUCT(*) AS cfg_state
       FROM state_ranked
       WHERE rn = 1
     )
     SELECT
       cfg.config_id,
       STRUCT(cfg.*) AS cfg_config,
       STRUCT(source.*) AS cfg_source,
       STRUCT(
         target.*,
         ARRAY_JOIN(
           TRANSFORM(
             MAP_ENTRIES(target.target_schema_definition),
             col -> CONCAT(
               'CAST(`', col.key, '` AS ', col.value, ') AS `', col.key, '`'
             )
           ),
           ', '
         ) AS target_schema_definition_sql
       ) AS cfg_target,
       cons.cfg_constraints AS cfg_constraints,
       st.cfg_state AS cfg_state
     FROM {control_catalog}.control.cfg_config cfg
     INNER JOIN {control_catalog}.control.cfg_source source
       ON source.config_id = cfg.config_id
     INNER JOIN {control_catalog}.control.cfg_target target
       ON target.config_id = cfg.config_id
     LEFT JOIN constraints_agg cons
       ON cons.config_id = cfg.config_id
     LEFT JOIN state_agg st
       ON st.config_id = cfg.config_id
     WHERE cfg.is_active = TRUE
       AND LOWER(cfg.system) = '" || v_system || "'
       AND UPPER(cfg.layer) = '" || v_layer || "'
       AND LOWER(cfg.config_name) = '" || v_config_name || "'";

  EXECUTE IMMEDIATE v_sql;

  SET v_cfg_count = (SELECT COUNT(*) FROM cfg);

  IF v_cfg_count = 0 THEN
    SIGNAL SQLSTATE '45000'
    SET MESSAGE_TEXT = 'proc_load_config: no config found';
  END IF;

  IF v_cfg_count > 1 THEN
    SIGNAL SQLSTATE '45000'
    SET MESSAGE_TEXT = 'proc_load_config: more than one row returned';
  END IF;
END;
""")